# Deep Q-Networks (DQN)

## Learning Objectives
1. Build a linear Q-network from scratch using numpy/torch for a toy 2D environment
2. Implement full DQN with experience replay buffer and target network on CartPole simulation
3. Extend to Double DQN and Dueling DQN architectures
4. Compare DQN variants on sample efficiency and stability

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import deque
import random
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## Level 1: Linear Q-Network on Toy 2D Environment

State = [x, y] normalized to [-1, 1]. Goal at (0.9, 0.9). 4 discrete actions.  
Train a single linear layer Q-network using torch.

In [ ]:
class Toy2DEnv:
    """Simple 2D navigation environment. State=[x,y] in [-1,1]. Goal at top-right."""

    def __init__(self):
        self.goal = np.array([0.9, 0.9])
        self.step_size = 0.2
        self.n_actions = 4   # 0=right, 1=left, 2=up, 3=down
        self.state_dim = 2
        self.action_map = np.array([[self.step_size, 0], [-self.step_size, 0],
                                    [0, self.step_size], [0, -self.step_size]])
        self.reset()

    def reset(self) -> np.ndarray:
        self.pos = np.array([-0.9, -0.9])  # Start at bottom-left
        return self.pos.copy()

    def step(self, action: int):
        self.pos = np.clip(self.pos + self.action_map[action], -1.0, 1.0)
        dist = np.linalg.norm(self.pos - self.goal)
        done = dist < 0.15
        reward = 1.0 if done else -0.05 * dist  # Dense reward: penalize distance
        return self.pos.copy(), reward, done


class LinearQNetwork(nn.Module):
    """Simple linear Q-network: state_dim -> n_actions."""

    def __init__(self, state_dim: int, n_actions: int):
        super().__init__()
        self.fc = nn.Linear(state_dim, n_actions)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


toy_env = Toy2DEnv()
linear_net = LinearQNetwork(toy_env.state_dim, toy_env.n_actions).to(device)
optimizer = optim.Adam(linear_net.parameters(), lr=1e-3)

# Simple training loop without replay buffer (Level 1: basic)
n_episodes = 300
gamma = 0.95
epsilon = 1.0
rewards_linear = []

for ep in range(n_episodes):
    state = toy_env.reset()
    total_reward = 0.0
    done = False
    steps = 0

    while not done and steps < 50:
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)

        # Epsilon-greedy action selection
        if np.random.rand() < epsilon:
            action = np.random.randint(toy_env.n_actions)
        else:
            with torch.no_grad():
                q_vals = linear_net(state_t)
                action = q_vals.argmax().item()

        next_state, reward, done = toy_env.step(action)

        # One-step TD update
        next_t = torch.FloatTensor(next_state).unsqueeze(0).to(device)
        with torch.no_grad():
            target = reward + gamma * linear_net(next_t).max() * (1 - done)

        q_pred = linear_net(state_t)[0, action]
        loss = F.mse_loss(q_pred, target.squeeze())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_reward += reward
        state = next_state
        steps += 1

    epsilon = max(0.05, epsilon * 0.99)
    rewards_linear.append(total_reward)

print(f'Linear Q-Net: mean reward (last 50 eps): {np.mean(rewards_linear[-50:]):.3f}')
print(f'Total parameters: {sum(p.numel() for p in linear_net.parameters())}')

## Level 2: Full DQN with Replay Buffer and Target Network

CartPole physics implemented from scratch (no gym).  
Key DQN components: experience replay buffer (deque, batch_size=32) + target network (update every 100 steps).

In [ ]:
def cartpole_step(state: np.ndarray, action: int, dt: float = 0.02):
    """CartPole physics. State: [x, x_dot, theta, theta_dot]. action: 0=left, 1=right."""
    x, x_dot, theta, theta_dot = state
    force = 10.0 if action == 1 else -10.0
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    temp = (force + 0.05 * theta_dot**2 * sin_t) / 1.1
    theta_acc = (9.8 * sin_t - cos_t * temp) / (0.5 * (4/3 - 0.1 * cos_t**2 / 1.1))
    x_acc = temp - 0.05 * theta_acc * cos_t / 1.1
    x += dt * x_dot
    x_dot += dt * x_acc
    theta += dt * theta_dot
    theta_dot += dt * theta_acc
    done = abs(x) > 2.4 or abs(theta) > 0.2
    reward = 1.0 if not done else 0.0
    return np.array([x, x_dot, theta, theta_dot]), reward, done


def cartpole_reset() -> np.ndarray:
    """Reset CartPole to small random initial state."""
    return np.random.uniform(-0.05, 0.05, size=4)


class ReplayBuffer:
    """Experience replay buffer. Stores (s, a, r, s', done) tuples."""

    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32))

    def __len__(self):
        return len(self.buffer)


class DQNNetwork(nn.Module):
    """DQN network: 2 hidden layers of 64 units each."""

    def __init__(self, state_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


def train_dqn(
    n_episodes: int = 300,
    batch_size: int = 32,
    buffer_capacity: int = 10000,
    target_update_freq: int = 100,
    gamma: float = 0.99,
    lr: float = 1e-4,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.05,
    epsilon_decay: float = 0.995,
    warmup_steps: int = 500,
) -> list:
    """Full DQN training loop on CartPole simulation."""
    online_net = DQNNetwork(4, 2).to(device)
    target_net = DQNNetwork(4, 2).to(device)
    target_net.load_state_dict(online_net.state_dict())
    target_net.eval()  # Target net is never trained directly

    optimizer = optim.Adam(online_net.parameters(), lr=lr)
    buffer = ReplayBuffer(buffer_capacity)

    epsilon = epsilon_start
    total_steps = 0
    episode_rewards = []

    for ep in range(n_episodes):
        state = cartpole_reset()
        total_reward = 0.0
        done = False

        while not done:
            # Epsilon-greedy action
            if np.random.rand() < epsilon:
                action = np.random.randint(2)
            else:
                with torch.no_grad():
                    s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                    action = online_net(s_t).argmax().item()

            next_state, reward, done = cartpole_step(state, action)
            buffer.push(state, action, reward, next_state, done)
            total_reward += reward
            state = next_state
            total_steps += 1

            # Only update once buffer has enough samples
            if len(buffer) >= warmup_steps:
                states, actions, rewards, next_states, dones = buffer.sample(batch_size)

                s = torch.FloatTensor(states).to(device)
                a = torch.LongTensor(actions).to(device)
                r = torch.FloatTensor(rewards).to(device)
                ns = torch.FloatTensor(next_states).to(device)
                d = torch.FloatTensor(dones).to(device)

                # Current Q values
                current_q = online_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

                # Target Q values (using target network — stable targets)
                with torch.no_grad():
                    target_q = r + gamma * target_net(ns).max(1)[0] * (1 - d)

                loss = F.smooth_l1_loss(current_q, target_q)  # Huber loss
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(online_net.parameters(), max_norm=10.0)
                optimizer.step()

                # Periodic hard update of target network
                if total_steps % target_update_freq == 0:
                    target_net.load_state_dict(online_net.state_dict())

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)

        if (ep + 1) % 50 == 0:
            print(f'Ep {ep+1:3d}: mean reward (last 20) = {np.mean(episode_rewards[-20:]):.1f}, '
                  f'epsilon = {epsilon:.3f}')

    return episode_rewards


torch.manual_seed(42); np.random.seed(42); random.seed(42)
dqn_rewards = train_dqn(n_episodes=300)
print(f'\nDQN final performance (last 50 eps): {np.mean(dqn_rewards[-50:]):.1f}')

## Real-World Example 1: Double DQN

Use online network to SELECT action, target network to EVALUATE it.  
Reduces overestimation bias without any architectural change.

In [ ]:
def train_double_dqn(
    n_episodes: int = 300,
    batch_size: int = 32,
    buffer_capacity: int = 10000,
    target_update_freq: int = 100,
    gamma: float = 0.99,
    lr: float = 1e-4,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.05,
    epsilon_decay: float = 0.995,
    warmup_steps: int = 500,
) -> tuple:
    """Double DQN: decouple action selection from action evaluation."""
    online_net = DQNNetwork(4, 2).to(device)
    target_net = DQNNetwork(4, 2).to(device)
    target_net.load_state_dict(online_net.state_dict())
    target_net.eval()

    optimizer = optim.Adam(online_net.parameters(), lr=lr)
    buffer = ReplayBuffer(buffer_capacity)

    epsilon = epsilon_start
    total_steps = 0
    episode_rewards = []
    q_value_log = []  # Track Q-value estimates to monitor overestimation

    for ep in range(n_episodes):
        state = cartpole_reset()
        total_reward = 0.0
        done = False

        while not done:
            if np.random.rand() < epsilon:
                action = np.random.randint(2)
            else:
                with torch.no_grad():
                    s_t = torch.FloatTensor(state).unsqueeze(0).to(device)
                    action = online_net(s_t).argmax().item()

            next_state, reward, done = cartpole_step(state, action)
            buffer.push(state, action, reward, next_state, done)
            total_reward += reward
            state = next_state
            total_steps += 1

            if len(buffer) >= warmup_steps:
                states, actions, rewards, next_states, dones = buffer.sample(batch_size)

                s = torch.FloatTensor(states).to(device)
                a = torch.LongTensor(actions).to(device)
                r = torch.FloatTensor(rewards).to(device)
                ns = torch.FloatTensor(next_states).to(device)
                d = torch.FloatTensor(dones).to(device)

                current_q = online_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    # Double DQN: SELECT with online, EVALUATE with target
                    best_actions = online_net(ns).argmax(1)  # Online net selects
                    next_q = target_net(ns).gather(1, best_actions.unsqueeze(1)).squeeze(1)  # Target evaluates
                    target_q = r + gamma * next_q * (1 - d)

                loss = F.smooth_l1_loss(current_q, target_q)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(online_net.parameters(), max_norm=10.0)
                optimizer.step()

                if total_steps % target_update_freq == 0:
                    target_net.load_state_dict(online_net.state_dict())

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)

        # Log max Q value at a fixed test state to monitor overestimation
        with torch.no_grad():
            test_s = torch.FloatTensor([0.0, 0.0, 0.05, 0.0]).unsqueeze(0).to(device)
            q_log = online_net(test_s).max().item()
        q_value_log.append(q_log)

    return episode_rewards, q_value_log


torch.manual_seed(42); np.random.seed(42); random.seed(42)
ddqn_rewards, ddqn_q_log = train_double_dqn(n_episodes=300)
print(f'Double DQN final performance (last 50 eps): {np.mean(ddqn_rewards[-50:]):.1f}')

## Real-World Example 2: Prioritized Experience Replay

Sample transitions with probability proportional to |TD error|.  
High-error transitions are more informative and should be replayed more often.

In [ ]:
class PrioritizedReplayBuffer:
    """Prioritized experience replay. Samples by TD error magnitude."""

    def __init__(self, capacity: int, alpha: float = 0.6):
        self.capacity = capacity
        self.alpha = alpha  # Priority exponent: 0=uniform, 1=full prioritization
        self.buffer = []
        self.priorities = np.zeros(capacity, dtype=np.float32)
        self.pos = 0
        self.max_priority = 1.0

    def push(self, transition):
        if len(self.buffer) < self.capacity:
            self.buffer.append(transition)
        else:
            self.buffer[self.pos] = transition
        # New transitions get max priority so they are sampled at least once
        self.priorities[self.pos] = self.max_priority
        self.pos = (self.pos + 1) % self.capacity

    def sample(self, batch_size: int, beta: float = 0.4):
        n = len(self.buffer)
        priorities = self.priorities[:n]
        probs = (priorities ** self.alpha)
        probs /= probs.sum()

        indices = np.random.choice(n, batch_size, p=probs, replace=False)
        batch = [self.buffer[i] for i in indices]

        # Importance sampling weights to correct for non-uniform sampling
        weights = (n * probs[indices]) ** (-beta)
        weights /= weights.max()  # Normalize

        states, actions, rewards, next_states, dones = zip(*batch)
        return (np.array(states), np.array(actions), np.array(rewards, dtype=np.float32),
                np.array(next_states), np.array(dones, dtype=np.float32),
                indices, weights.astype(np.float32))

    def update_priorities(self, indices: np.ndarray, td_errors: np.ndarray):
        """Update priorities after computing TD errors."""
        for idx, err in zip(indices, td_errors):
            # Add small epsilon to avoid zero priority
            priority = abs(float(err)) + 1e-5
            self.priorities[idx] = priority
            self.max_priority = max(self.max_priority, priority)

    def __len__(self):
        return len(self.buffer)


def train_per_dqn(n_episodes: int = 300, batch_size: int = 32, warmup_steps: int = 500) -> list:
    """DQN with Prioritized Experience Replay (PER)."""
    online_net = DQNNetwork(4, 2).to(device)
    target_net = DQNNetwork(4, 2).to(device)
    target_net.load_state_dict(online_net.state_dict()); target_net.eval()

    optimizer = optim.Adam(online_net.parameters(), lr=1e-4)
    buffer = PrioritizedReplayBuffer(10000)
    epsilon = 1.0; total_steps = 0; rewards = []

    for ep in range(n_episodes):
        state = cartpole_reset(); total_reward = 0.0; done = False
        while not done:
            action = np.random.randint(2) if np.random.rand() < epsilon else \
                online_net(torch.FloatTensor(state).unsqueeze(0).to(device)).argmax().item()
            ns, r, done = cartpole_step(state, action)
            buffer.push((state, action, r, ns, done))
            total_reward += r; state = ns; total_steps += 1

            if len(buffer) >= warmup_steps:
                states_b, acts_b, rwds_b, nss_b, ds_b, idxs, wts = buffer.sample(batch_size, beta=0.4)

                s = torch.FloatTensor(states_b).to(device)
                a = torch.LongTensor(acts_b).to(device)
                r_t = torch.FloatTensor(rwds_b).to(device)
                ns_t = torch.FloatTensor(nss_b).to(device)
                d_t = torch.FloatTensor(ds_b).to(device)
                w_t = torch.FloatTensor(wts).to(device)

                curr_q = online_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    next_q = r_t + 0.99 * target_net(ns_t).max(1)[0] * (1 - d_t)

                td_errors = (next_q - curr_q).detach().cpu().numpy()
                buffer.update_priorities(idxs, td_errors)

                # IS-weighted loss
                loss = (w_t * F.smooth_l1_loss(curr_q, next_q, reduction='none')).mean()
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(online_net.parameters(), 10.0)
                optimizer.step()

                if total_steps % 100 == 0:
                    target_net.load_state_dict(online_net.state_dict())

        epsilon = max(0.05, epsilon * 0.995)
        rewards.append(total_reward)

    return rewards


torch.manual_seed(42); np.random.seed(42); random.seed(42)
per_rewards = train_per_dqn(n_episodes=300)
print(f'PER-DQN final performance (last 50 eps): {np.mean(per_rewards[-50:]):.1f}')

## Real-World Example 3: Dueling DQN Architecture

Separate the Q-network into Value stream V(s) and Advantage stream A(s,a).  
Q(s,a) = V(s) + A(s,a) - mean_a A(s,a). Learns V(s) faster for states where actions do not matter.

In [ ]:
class DuelingDQNNetwork(nn.Module):
    """Dueling DQN: separate value and advantage streams."""

    def __init__(self, state_dim: int, n_actions: int, hidden: int = 64):
        super().__init__()
        # Shared feature extraction
        self.feature = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
        )
        # Value stream: scalar V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
        # Advantage stream: vector A(s, a) for each action
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.feature(x)
        value = self.value_stream(features)        # Shape: (batch, 1)
        advantage = self.advantage_stream(features)  # Shape: (batch, n_actions)
        # Combine: subtract mean advantage to ensure identifiability
        q = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q


def train_dueling_dqn(n_episodes: int = 300, batch_size: int = 32, warmup_steps: int = 500) -> list:
    """Dueling DQN training on CartPole simulation."""
    online_net = DuelingDQNNetwork(4, 2).to(device)
    target_net = DuelingDQNNetwork(4, 2).to(device)
    target_net.load_state_dict(online_net.state_dict()); target_net.eval()

    optimizer = optim.Adam(online_net.parameters(), lr=1e-4)
    buffer = ReplayBuffer(10000)
    epsilon = 1.0; total_steps = 0; rewards = []

    for ep in range(n_episodes):
        state = cartpole_reset(); total_reward = 0.0; done = False
        while not done:
            if np.random.rand() < epsilon:
                action = np.random.randint(2)
            else:
                with torch.no_grad():
                    action = online_net(torch.FloatTensor(state).unsqueeze(0).to(device)).argmax().item()
            ns, r, done = cartpole_step(state, action)
            buffer.push(state, action, r, ns, done)
            total_reward += r; state = ns; total_steps += 1

            if len(buffer) >= warmup_steps:
                states_b, acts_b, rwds_b, nss_b, ds_b = buffer.sample(batch_size)
                s = torch.FloatTensor(states_b).to(device)
                a = torch.LongTensor(acts_b).to(device)
                r_t = torch.FloatTensor(rwds_b).to(device)
                ns_t = torch.FloatTensor(nss_b).to(device)
                d_t = torch.FloatTensor(ds_b).to(device)

                curr_q = online_net(s).gather(1, a.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    # Double DQN + Dueling
                    best_a = online_net(ns_t).argmax(1)
                    tgt_q = r_t + 0.99 * target_net(ns_t).gather(1, best_a.unsqueeze(1)).squeeze(1) * (1 - d_t)

                loss = F.smooth_l1_loss(curr_q, tgt_q)
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(online_net.parameters(), 10.0)
                optimizer.step()
                if total_steps % 100 == 0:
                    target_net.load_state_dict(online_net.state_dict())

        epsilon = max(0.05, epsilon * 0.995)
        rewards.append(total_reward)

    return rewards


torch.manual_seed(42); np.random.seed(42); random.seed(42)
dueling_rewards = train_dueling_dqn(n_episodes=300)
print(f'Dueling DQN final performance (last 50 eps): {np.mean(dueling_rewards[-50:]):.1f}')

## Comparison: DQN vs Double DQN vs Dueling DQN

Compare all three variants on CartPole simulation.  
Also plot Q-value estimates to visualize overestimation reduction.

In [ ]:
def smooth(arr, w=15):
    return np.convolve(arr, np.ones(w) / w, mode='valid')


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Learning curves ---
ax = axes[0]
ax.plot(smooth(dqn_rewards), label='DQN', color='red')
ax.plot(smooth(ddqn_rewards), label='Double DQN', color='blue')
ax.plot(smooth(dueling_rewards), label='Dueling DQN', color='green')
ax.plot(smooth(per_rewards), label='PER-DQN', color='purple')
ax.set_title('CartPole: Learning Curves')
ax.set_xlabel('Episode'); ax.set_ylabel('Episode Reward (smoothed)')
ax.legend()

# --- Plot 2: Q-value overestimation comparison ---
ax = axes[1]
ax.plot(smooth(ddqn_q_log, 10), label='Double DQN Q-value', color='blue')
ax.axhline(200, color='black', linestyle='--', alpha=0.5, label='Max possible (200 steps)')
ax.set_title('Q-Value Estimates: Overestimation Check')
ax.set_xlabel('Episode'); ax.set_ylabel('Max Q(s_test)')
ax.legend()

# --- Plot 3: Final performance box plot ---
ax = axes[2]
final_rewards = [
    dqn_rewards[-50:], ddqn_rewards[-50:],
    dueling_rewards[-50:], per_rewards[-50:]
]
ax.boxplot(final_rewards, labels=['DQN', 'Double', 'Dueling', 'PER'])
ax.set_title('Final 50 Episodes Distribution')
ax.set_ylabel('Episode Reward')

plt.suptitle('DQN Variants Comparison on CartPole', fontsize=13)
plt.tight_layout()
plt.savefig('/tmp/dqn_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/dqn_comparison.png')

print('\n=== Final Performance Summary ===')
for name, rw in [('DQN', dqn_rewards), ('Double DQN', ddqn_rewards),
                  ('Dueling DQN', dueling_rewards), ('PER DQN', per_rewards)]:
    print(f'{name:<15}: mean={np.mean(rw[-50:]):6.1f}, max={np.max(rw[-50:]):6.1f}')

## Key Takeaways

**Core idea:** DQN extends Q-learning to large state spaces using a neural network, stabilized by two key tricks: experience replay (breaks temporal correlation) and a target network (prevents chasing a moving target).

**Variants and when to use:**

| Algorithm | Key Improvement | Cost | Best for |
|-----------|---------------|------|----------|
| DQN | Replay + target net | Baseline | Any discrete control |
| Double DQN | Reduces overestimation | None | When Q > actual return |
| Dueling DQN | Faster V(s) learning | Slightly wider net | Many irrelevant actions |
| PER | Better sample efficiency | Priority queue | Sparse rewards |

**Common failure modes:**
- Missing done flag: Q-values inflate unboundedly — always check terminal state handling
- Target net updated too often (C=1): equivalent to no target net; unstable training
- Buffer too small: highly correlated samples; agent forgets early experience

**Related concepts:**
- [06-q-learning](./06-q-learning.ipynb) — tabular foundation DQN extends
- [09-policy-gradient](./09-policy-gradient.ipynb) — alternative direct policy optimization
- [10-actor-critic](./10-actor-critic.ipynb) — combines value and policy networks

## Exercises

1. **Soft target update:** Replace the hard target network update (copy every C=100 steps) with a Polyak soft update: `theta_target = 0.005*theta + 0.995*theta_target` applied every step. Compare convergence stability.
2. **Buffer size sweep:** Train DQN with buffer sizes 1000, 5000, 20000. Plot how long convergence takes as a function of buffer size. What is the minimum buffer that still converges reliably?
3. **Dueling visualization:** After training Dueling DQN, extract and plot V(s) and A(s,a) separately across a grid of CartPole states. Verify that V(s) varies smoothly while A(s,a) captures action-level differences.
4. **Overestimation analysis:** Log max Q(s,a) every episode for both DQN and Double DQN. Compare to the actual mean return (ground truth). Quantify the overestimation ratio.